# Imports/Settings

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path
import joblib
import json

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Сторонние библиотеки
import pandas as pd
import gradio as gr

# 3. Локальные модули
from core.models import get_model
from core.utils import load_hydra_config

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Error Analysis")

Проект: outsource_project_name | Режим: Error Analysis


# Artefacts Loading

In [3]:
# Пути к сохраненным объектам
models_dir = PROJECT_ROOT / cfg.paths.models_dir
model_version = cfg.model.model_version
preprocessor_version = cfg.data.tabular.preprocessing_version
schema_version = cfg.data.tabular.features_version
model_name = cfg.model.name

print("Поднятие модели в память...")
prep = joblib.load(models_dir / f"preprocessing_v{preprocessor_version}.pkl")

feature_schema_path = models_dir / f"feature_schema_v{schema_version}.json"
with open(feature_schema_path, "r", encoding="utf-8") as f:
    features_schema = json.load(f)

model = get_model(cfg)
model.load(str(models_dir / f"{model_name}_v{model_version}.cbm"))

print("Модель готова к инференсу!")

Поднятие модели в память...
Модель готова к инференсу!


# Prediction Function

In [4]:
# ============================================================================
# ШАГ 1: АВТОМАТИЧЕСКИЙ РАЗБОР ФЛАТ-СХЕМЫ ПО ТИПАМ ДАННЫХ PANDAS
# ============================================================================
required_inputs = list(prep.feature_names_in_)

# Автоматически определяем скрытые служебные колонки из конфига Hydra
hidden_cols = list(cfg.data.tabular.get('drop_cols', []))
# Железный фолбэк для системных ID веб-аналитики
for tech_id in ['session_id', 'client_id', 'rn']:
    if tech_id in required_inputs and tech_id not in hidden_cols:
        hidden_cols.append(tech_id)

# Считываем списки категорий и чисел прямо из типов в features_schema
categorical_cols = [col for col, dtype in features_schema.items() if dtype == 'object']
numeric_cols = [col for col, dtype in features_schema.items() if 'int' in dtype or 'float' in dtype]

# Пытаемся вытащить обученные топ-значения категорий из препроцессора для Dropdown
predefined_cat_maps = {}
if hasattr(prep, 'named_steps'):
    for step_name, step_obj in prep.named_steps.items():
        if hasattr(step_obj, 'top_categories_map_') and step_obj.top_categories_map_:
            predefined_cat_maps = step_obj.top_categories_map_
            break

print(f"Пайплайн успешно инициализирован. Всего фичей: {len(required_inputs)}")
print(f"Найдено категориальных признаков по JSON: {len(categorical_cols)}")
print(f"Найдено числовых признаков по JSON: {len(numeric_cols)}")
print(f"Авто-скрыто технических ID: {len(hidden_cols)}")

# ============================================================================
# ШАГ 2: ДЕКЛАРАТИВНАЯ СБОРКА UI НА ОСНОВЕ ТИПОВ СХЕМЫ (ZERO HARDCODE)
# ============================================================================
inputs = []

for col in required_inputs:
    # 1. Системные невидимые поля
    if col in hidden_cols:
        inputs.append(gr.Textbox(value="1", label=f"Служебный ID: {col}", visible=False))
        
    # 2. Строковые / Категориальные признаки (dtype == 'object')
    elif col in categorical_cols:
        # Проверяем, есть ли готовый список категорий в памяти препроцессора
        choices = list(predefined_cat_maps.get(col, []))
        
        # Если препроцессор пустой (грязный бейзлайн) — даем свободный текстовый ввод,
        # модель не упадет, так как на инференсе тип будет приведен к str!
        if not choices:
            inputs.append(gr.Textbox(value="Unknown", label=f"Категория (Текст): {col}"))
        else:
            if 'other_collapsed' not in choices:
                choices.append('other_collapsed')
            inputs.append(gr.Dropdown(choices=choices, value=choices[0], label=f"Категория: {col}"))
            
    # 3. Числовые признаки (dtype == 'int64' / 'float64')
    elif col in numeric_cols:
        # Выставляем безопасные лимиты слайдера. Для часов делаем 23, для остального универсально 100
        max_v = 23 if 'hour' in col else 100
        inputs.append(gr.Slider(minimum=0, maximum=max_v, step=1, value=1, label=f"Число: {col}"))
        
    # 4. Бинарные флаги (на случай непредвиденных типов)
    else:
        if col.startswith('is_') or col.startswith('has_'):
            inputs.append(gr.Checkbox(label=f"Флаг: {col}", value=False))
        else:
            inputs.append(gr.Textbox(value="0", label=f"Поле (Вне явных типов): {col}"))
            
# ============================================================================
# ШАГ 3: ДИНАМИЧЕСКИЙ ИНФЕРЕНС С ПРИВЯЗКОЙ К КОМПОНЕНТАМ UI
# ============================================================================
def predict_conversion_dynamic(*args):
    # Собираем DataFrame
    input_df = pd.DataFrame({col: [val] for col, val in zip(required_inputs, args)})
    
    # Конвертируем типы строго по типу компонента Gradio, созданного на основе JSON
    for col, component in zip(required_inputs, inputs):
        if isinstance(component, gr.Dropdown) or (isinstance(component, gr.Textbox) and col in categorical_cols):
            input_df[col] = input_df[col].astype(str)
        elif isinstance(component, gr.Slider) or (isinstance(component, gr.Textbox) and col in numeric_cols):
            input_df[col] = pd.to_numeric(input_df[col], errors='coerce').fillna(0).astype(int)
        elif isinstance(component, gr.Checkbox):
            input_df[col] = input_df[col].astype(int)
            
    # Передаем DataFrame в препроцессинг
    clean_data = prep.transform(input_df)
    
    task_type = cfg.data.tabular.get('task_type', 'binary')
    if hasattr(model, 'predict_proba') and task_type == 'binary':
        prob = model.predict_proba(clean_data)[0][1]
        result_text = f" Вероятность: {prob * 100:.1f}%\n"
        return result_text + ("Вердикт: Горячий лид" if prob > 0.5 else "Вердикт: Низкий потенциал 💤")
    else:
        pred = model.predict(clean_data)[0]
        return f" Результат модели: {pred:.2f}"

Пайплайн успешно инициализирован. Всего фичей: 19
Найдено категориальных признаков по JSON: 17
Найдено числовых признаков по JSON: 2
Авто-скрыто технических ID: 4


# Gradio

In [5]:
output = gr.Textbox(label="Результат работы модели")

demo = gr.Interface(
    fn=predict_conversion_dynamic,
    inputs=inputs,
    outputs=output,
    title=f"🔮 {cfg.project_name} - Демонстрация Модели",
    description="Интерфейс подстраивается под структуру TabularPreprocessor динамически.",
    theme="default"
)

demo.launch(share=False)

c:\sber_autopodpiska\.venv\lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


c:\sber_autopodpiska\.venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Created dataset file at: .gradio\flagged\dataset1.csv
